# OAT 3 — Encontro 2 (07/10/2026)
## Grid Search e sintonia de hiperparâmetros

In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

## Preparar os dados e os atributos temporais

In [2]:
df = pd.read_csv("../dados/mecaniqa_dataset.csv", parse_dates=["Data"])
df = df.sort_values("Data").set_index("Data")

dados = df.copy()
dados["Lag_1"] = dados["Trocas_Oleo"].shift(1)
dados["Lag_7"] = dados["Trocas_Oleo"].shift(7)
dados["Media_7"] = dados["Trocas_Oleo"].shift(1).rolling(7).mean()
dados["Dia_Semana"] = dados.index.dayofweek

colunas = ["Lag_1", "Lag_7", "Media_7", "Dia_Semana"]
dados = dados.dropna(subset=colunas + ["Trocas_Oleo"])
X = dados[colunas]
y = dados["Trocas_Oleo"]
tscv = TimeSeriesSplit(n_splits=5)
print("Registros preparados:", len(dados))
print("Divisões temporais:", tscv.n_splits)

Registros preparados: 711
Divisões temporais: 5


## Configurar os modelos e os hiperparâmetros

In [3]:
modelos = {
    "Árvore de Decisão": (
        DecisionTreeRegressor(random_state=42),
        {
            "max_depth": [3, 5, 10, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    ),
    "Random Forest": (
        RandomForestRegressor(random_state=42),
        {
            "n_estimators": [50, 100, 200],
            "max_depth": [5, 10, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    )
}

## Executar Grid Search e comparar com o modelo padrão


In [4]:
resultados = []

for nome, (modelo, parametros) in modelos.items():
    mae_padrao = -cross_val_score(
        modelo, X, y, scoring="neg_mean_absolute_error",
        cv=tscv, n_jobs=-1
    ).mean()

    grid = GridSearchCV(
        modelo, parametros, scoring="neg_mean_absolute_error",
        cv=tscv, n_jobs=-1
    )
    inicio = time.perf_counter()
    grid.fit(X, y)
    tempo = time.perf_counter() - inicio

    mae_tunado = -grid.best_score_
    reducao = (mae_padrao - mae_tunado) / mae_padrao * 100

    print(f"\n{nome}")
    print("Melhores parâmetros:", grid.best_params_)
    print(f"MAE padrão: {mae_padrao:.2f}")
    print(f"MAE tunado: {mae_tunado:.2f}")
    print(f"Redução do MAE: {reducao:.2f}%")

    resultados.append({
        "Modelo": nome,
        "MAE padrão": mae_padrao,
        "MAE Grid Search": mae_tunado,
        "Redução MAE (%)": reducao,
        "Tempo Grid Search (s)": tempo,
        "Melhores parâmetros": grid.best_params_
    })

tabela_grid = pd.DataFrame(resultados)
display(tabela_grid.round(2))


Árvore de Decisão
Melhores parâmetros: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}
MAE padrão: 5.40
MAE tunado: 4.75
Redução do MAE: 11.92%

Random Forest
Melhores parâmetros: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 50}
MAE padrão: 4.73
MAE tunado: 4.44
Redução do MAE: 6.03%


,Modelo,MAE padrão,MAE Grid Search,Redução MAE (%),Tempo Grid Search (s),Melhores parâmetros
0,Árvore de Decisão,5.40,4.75,11.92,1.86,"{'max_depth': 5, 'min_samples_leaf': 1, 'min_s..."
1,Random Forest,4.73,4.44,6.03,170.68,"{'max_depth': 10, 'min_samples_leaf': 2, 'min_..."
